##### when & otherwise

**1) Create a Sample DataFrame**

In [0]:
from pyspark.sql.functions import when, col, lit, concat

In [0]:
# Create a sample DataFrame
data = [
    (1, "Janardan", 5000),
    (2, "Alia", 7000),
    (3, "Bindu", 3000),
    (4, "Charuni", 10000),
    (5, "Dravid", 4000),
    (6, "Tarun", 1500),
    (7, "Seetha", 1000)
]

columns = ["emp_id", "emp_name", "salary"]

df = spark.createDataFrame(data, columns)
display(df)

emp_id,emp_name,salary
1,Janardan,5000
2,Alia,7000
3,Bindu,3000
4,Charuni,10000
5,Dravid,4000
6,Tarun,1500
7,Seetha,1000


**a) Using when and otherwise for Conditional Columns**
- Let’s add a new column **salary_grade** based on the **employee’s salary**.

In [0]:
df2 = df.withColumn(
    "salary_grade",
    when(col("salary") >= 8000, "High")
    .when((col("salary") >= 5000) & (col("salary") < 8000), "Medium")
    .otherwise("Low")
)

display(df2)

emp_id,emp_name,salary,salary_grade
1,Janardan,5000,Medium
2,Alia,7000,Medium
3,Bindu,3000,Low
4,Charuni,10000,High
5,Dravid,4000,Low
6,Tarun,1500,Low
7,Seetha,1000,Low


**b) Create a bonus column based on salary**

In [0]:
df3 = df.withColumn(
    "bonus",
    when(col("salary") > 8000, col("salary") * 0.2)
    .when(col("salary") >= 5000, col("salary") * 0.1)
    .otherwise(col("salary") * 0.05)
)

display(df3)

emp_id,emp_name,salary,bonus
1,Janardan,5000,500.0
2,Alia,7000,700.0
3,Bindu,3000,150.0
4,Charuni,10000,2000.0
5,Dravid,4000,200.0
6,Tarun,1500,75.0
7,Seetha,1000,50.0


**c) Using Multiple Conditions with when and Logical Operators**

In [0]:
df4 = df.withColumn(
    "category",
    when((col("salary") >= 8000) & (col("emp_name").startswith("C")), "Top Performer")
    .when((col("salary") < 8000) & (col("salary") >= 4000), "High Earner")
    .when((col("salary") < 4000) & (col("salary") >= 2000), "Average")
    .otherwise("Needs Improvement")
)

display(df4)

emp_id,emp_name,salary,category
1,Janardan,5000,High Earner
2,Alia,7000,High Earner
3,Bindu,3000,Average
4,Charuni,10000,Top Performer
5,Dravid,4000,High Earner
6,Tarun,1500,Needs Improvement
7,Seetha,1000,Needs Improvement


**d) Using when inside select()**

In [0]:
df.select(
    "emp_name",
    "salary",
    when(col("salary") >= 6000, "Eligible for Promotion")
    .otherwise("Not Eligible")
    .alias("promotion_status")
).display()

emp_name,salary,promotion_status
Janardan,5000,Not Eligible
Alia,7000,Eligible for Promotion
Bindu,3000,Not Eligible
Charuni,10000,Eligible for Promotion
Dravid,4000,Not Eligible
Tarun,1500,Not Eligible
Seetha,1000,Not Eligible


**2) Create a DataFrame with NULL values**

In [0]:
data = [
    (1, "Janardan", 5000),
    (2, "Alia", None),
    (3, "Bindu", 3000),
    (4, "Charuni", 10000),
    (5, None, 4000),
    (6, "Dravid", None)
]

columns = ["emp_id", "emp_name", "salary"]

df_null = spark.createDataFrame(data, columns)
display(df_null)

emp_id,emp_name,salary
1,Janardan,5000
2,Alia,null
3,Bindu,3000
4,Charuni,10000
5,null,4000
6,Dravid,null


**a) Handle NULL values using when() and otherwise()**

In [0]:
df2 = df_null.withColumn(
    "salary_status",
    when(col("salary").isNull(), "Salary Not Available")
    .when(col("salary") >= 8000, "High")
    .when((col("salary") >= 4000) & (col("salary") < 8000), "Medium")
    .otherwise("Low")
)

display(df2)

emp_id,emp_name,salary,salary_status
1,Janardan,5000,Medium
2,Alia,null,Salary Not Available
3,Bindu,3000,Low
4,Charuni,10000,High
5,null,4000,Medium
6,Dravid,null,Salary Not Available


**b) Replace NULL names and create greeting message**

In [0]:
df3 = df_null \
.withColumn("emp_name_filled", when(col("emp_name").isNull(), lit("Unknown Employee")).otherwise(col("emp_name"))) \
.withColumn("greeting", when(col("emp_name").isNull(), lit("Hello, Employee!")).otherwise(concat(col("emp_name"), lit(", welcome!"))))

display(df3)

emp_id,emp_name,salary,emp_name_filled,greeting
1,Janardan,5000,Janardan,"Janardan, welcome!"
2,Alia,null,Alia,"Alia, welcome!"
3,Bindu,3000,Bindu,"Bindu, welcome!"
4,Charuni,10000,Charuni,"Charuni, welcome!"
5,null,4000,Unknown Employee,"Hello, Employee!"
6,Dravid,null,Dravid,"Dravid, welcome!"


**c) Handling both NULL and numeric logic together**

In [0]:
df4 = df_null.withColumn("bonus", when(col("salary").isNull(), lit(0))
                                  .when(col("salary") >= 8000, col("salary") * 0.2)
                                  .when(col("salary") >= 4000, col("salary") * 0.1)
                                  .otherwise(col("salary") * 0.05)
)

display(df4)

emp_id,emp_name,salary,bonus
1,Janardan,5000,500.0
2,Alia,null,0.0
3,Bindu,3000,150.0
4,Charuni,10000,2000.0
5,null,4000,400.0
6,Dravid,null,0.0
